# KMRF Model - Batch Training Notebook

This notebook trains KMRF models across multiple assets in batch mode.

**Output Structure:**
```
saved_models/KMRF/
├── adapted_labels/
│   ├── us_equity/
│   ├── commodity/
│   └── us_treasury/
└── original_labels/
    └── ...
```

#### Imports

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
import traceback
import json

warnings.filterwarnings('ignore')

import kmrf as kmrf

%matplotlib inline

## Asset Lists

Define the assets to train models for:

In [21]:
# US Equity Assets
us_equity_assets = [
    # Major Indices ETFs
    'SPDR S&P 500 ETF',  # SPY
    'Invesco QQQ Trust',  # QQQ
    'iShares Russell 2000 ETF',  # IWM
    'SPDR Dow Jones Industrial Average ETF',  # DIA
    
    # Sector ETFs (Select Sector SPDRs)
    'Energy Select Sector SPDR',  # XLE
    'Financial Select Sector SPDR',  # XLF
    'Utilities Select Sector SPDR',  # XLU
    'Industrial Select Sector SPDR',  # XLI
    'Health Care Select Sector SPDR',  # XLV
    'Technology Select Sector SPDR',  # XLK
    'Materials Select Sector SPDR',  # XLB
    'Consumer Discretionary Select Sector SPDR',  # XLY
    'Consumer Staples Select Sector SPDR',  # XLP
    
    # Style ETFs
    'iShares S&P 500 Growth ETF',  # IVW
    'iShares S&P 500 Value ETF',  # IVE
    'iShares Russell 2000 Growth ETF',  # IWF
    'iShares Russell 2000 Value ETF',  # IWD
]

# US Treasury Assets
us_treasury_assets = [
    'SPDR Bloomberg 1-3 Month T-Bill ETF',  # BIL
    'iShares 1-3 Year Treasury Bond ETF',  # SHY
    'iShares 7-10 Year Treasury Bond ETF',  # IEF
]

# International Equity ETFs
int_equity_assets = [
    'Vanguard Total International Stock ETF',  # VXUS
    'Vanguard FTSE Developed Markets ETF',  # VEA
    'Vanguard FTSE Emerging Markets ETF',  # VWO
    'Vanguard FTSE Europe ETF',  # VGK
    'Vanguard FTSE Pacific ETF',  # VPL
    'iShares China Large-Cap ETF',  # FXI
    'iShares MSCI Japan ETF',  # EWJ
    'iShares MSCI India ETF',  # INDA
]

# Commodity Assets
commodity_assets = [
    # 'Aluminum Futures', 
    'Gold Futures', 'Wheat Futures', 'Corn Futures',
    'Copper', 'Sugar', 'Soybean Futures', 'Live Cattle Futures',
    'Natural Gas', 'Coffee', 'Brent Crude Oil'
]

# Map asset classes to their asset lists
ASSET_GROUPS = {
    'us_equity': us_equity_assets,
    'us_treasury': us_treasury_assets,
    'int_equity': int_equity_assets,
    'commodity': commodity_assets,
}

## *** Configuration ***

Set batch training parameters:

In [22]:
# ============================================================================
# BATCH TRAINING CONFIGURATION
# ============================================================================

# Select which asset classes to train
ASSET_CLASSES_TO_TRAIN = ['commodity']  # Options: 'us_equity', 'int_equity', 'commodity'

# Classification type
CLASSIFICATION_TYPE = 'original'  # 'original' (4 regimes) or 'adapted' (3 regimes)

# Feature selection
USE_BORUTA = True  # WARNING: Boruta is slow
BORUTA_MAX_ITER = 100  # Max iterations for Boruta

# Training/validation/test split dates
END_DATE = '20190101'  # Data up to this date for training
VALIDATION_START = '2019-04-01'
VALIDATION_END = '2019-09-30'
TEST_START = '2020-01-01'

# Other parameters
RANDOM_SEED = 1010
INCLUDE_MACRO = True  # Include macroeconomic features

# Error handling
CONTINUE_ON_ERROR = True  # If True, continue training other assets if one fails
SAVE_SUCCESSFUL_ONLY = True  # Only save models that trained successfully

# Verbose output
VERBOSE = True

## Batch Training Function

In [23]:
def train_kmrf_batch(
    asset_names: List[str],
    asset_class: str,
    classification_type: str = 'adapted',
    end_date: str = '20190101',
    validation_start: str = '2019-04-01',
    validation_end: str = '2019-09-30',
    test_start: str = '2020-01-01',
    use_boruta: bool = False,
    boruta_params: Optional[Dict] = None,
    include_macro: bool = True,
    random_seed: int = 42,
    continue_on_error: bool = True,
    save_successful_only: bool = True,
    verbose: bool = True
) -> Dict[str, Dict]:
    """
    Train KMRF models for multiple assets in batch mode.
    
    Parameters
    ----------
    asset_names : List[str]
        List of asset names to train models for
    asset_class : str
        Asset class ('us_equity', 'us_treasury', 'int_equity', 'commodity')
    classification_type : str, default='adapted'
        Classification type ('original' or 'adapted')
    end_date : str
        Training data end date
    validation_start : str
        Validation period start date
    validation_end : str
        Validation period end date
    test_start : str
        Test period start date
    use_boruta : bool, default=False
        Whether to use Boruta feature selection
    boruta_params : dict, optional
        Parameters for Boruta (default: {'max_iter': 100})
    include_macro : bool, default=True
        Whether to include macroeconomic features
    random_seed : int, default=42
        Random seed for reproducibility
    continue_on_error : bool, default=True
        Whether to continue if one asset fails
    save_successful_only : bool, default=True
        Only save successfully trained models
    verbose : bool, default=True
        Print detailed progress
        
    Returns
    -------
    Dict[str, Dict]
        Dictionary with training results and metadata for each asset
    """
    
    if boruta_params is None:
        boruta_params = {'max_iter': 100}
    
    results = {}
    successful_models = []
    failed_models = []
    
    total_assets = len(asset_names)
    
    print(f"\n{'='*80}")
    print(f"BATCH TRAINING KMRF MODELS")
    print(f"{'='*80}")
    print(f"Asset Class: {asset_class}")
    print(f"Classification Type: {classification_type}")
    print(f"Total Assets: {total_assets}")
    print(f"Use Boruta: {use_boruta}")
    print(f"Include Macro: {include_macro}")
    print(f"Training Period: up to {end_date}")
    print(f"Validation: {validation_start} to {validation_end}")
    print(f"Test: {test_start} onwards")
    print(f"{'='*80}\n")
    
    for idx, asset_name in enumerate(asset_names, 1):
        print(f"\n{'='*80}")
        print(f"[{idx}/{total_assets}] Processing: {asset_name}")
        print(f"{'='*80}")
        
        start_time = datetime.now()
        
        try:
            # Initialize model
            if verbose:
                print(f"\n1. Initializing KMRF model...")
            
            model = kmrf.KMRF(
                asset_name=asset_name,
                asset_class=asset_class,
                end_date=end_date,
                use_ready_data=True,
                validation_start=validation_start,
                validation_end=validation_end,
                test_start=test_start,
                random_seed=random_seed,
                classification_type=classification_type
            )
            
            # Load data
            if verbose:
                print(f"2. Loading data...")
            data = model.load_data()
            features = model.get_features()
            
            if verbose:
                print(f"   Data shape: {data.shape}")
                print(f"   Features shape: {features.shape}")
            
            # Load labels
            if verbose:
                print(f"3. Loading KAMA+MSR labels...")
            model.load_kama_msr_labels()
            
            if classification_type == 'adapted':
                if verbose:
                    print(f"   Adapting to 3-class labels...")
                model.adapt_regime_labels()
            
            # Prepare training data
            if verbose:
                print(f"4. Preparing training data...")
                if use_boruta:
                    print(f"   Running Boruta feature selection (this may take 30+ minutes)...")
            
            X_train, y_train = model.prepare_training_data(
                include_macro=include_macro,
                select_features=use_boruta,
                boruta_params=boruta_params,
                split_data=True
            )
            
            if verbose:
                print(f"   Training samples: {len(X_train)}")
                print(f"   Features: {X_train.shape[1]}")
                if model.selected_features:
                    print(f"   Selected features: {len(model.selected_features)}")
            
            # Train model
            if verbose:
                print(f"5. Training Random Forest model...")
            
            model.fit(X_train, y_train)
            
            # Save model
            if verbose:
                print(f"6. Saving model...")
            
            model_path = model.save_model(boruta_used=use_boruta)
            
            # Record success
            end_time = datetime.now()
            duration = (end_time - start_time).total_seconds()
            
            results[asset_name] = {
                'status': 'success',
                'model_path': str(model_path),
                'n_features': X_train.shape[1],
                'n_selected_features': len(model.selected_features) if model.selected_features else None,
                'n_train_samples': len(X_train),
                'n_test_samples': len(model.X_test) if model.X_test is not None else None,
                'boruta_used': use_boruta,
                'duration_seconds': duration,
                'timestamp': datetime.now().isoformat()
            }
            
            successful_models.append(asset_name)
            
            print(f"\n✓ SUCCESS: {asset_name}")
            print(f"  Model saved to: {model_path}")
            print(f"  Training time: {duration/60:.1f} minutes")
            
        except Exception as e:
            # Record failure
            end_time = datetime.now()
            duration = (end_time - start_time).total_seconds()
            
            error_msg = str(e)
            error_trace = traceback.format_exc()
            
            results[asset_name] = {
                'status': 'failed',
                'error': error_msg,
                'error_trace': error_trace,
                'duration_seconds': duration,
                'timestamp': datetime.now().isoformat()
            }
            
            failed_models.append(asset_name)
            
            print(f"\n✗ FAILED: {asset_name}")
            print(f"  Error: {error_msg}")
            
            if verbose:
                print(f"\n  Full traceback:")
                print(error_trace)
            
            if not continue_on_error:
                print(f"\nStopping batch training due to error.")
                break
    
    # Print summary
    print(f"\n{'='*80}")
    print(f"BATCH TRAINING COMPLETE")
    print(f"{'='*80}")
    print(f"Total Assets: {total_assets}")
    print(f"Successful: {len(successful_models)}")
    print(f"Failed: {len(failed_models)}")
    
    if successful_models:
        print(f"\n✓ Successfully trained models:")
        for name in successful_models:
            duration = results[name]['duration_seconds']
            print(f"  - {name} ({duration/60:.1f} min)")
    
    if failed_models:
        print(f"\n✗ Failed models:")
        for name in failed_models:
            print(f"  - {name}: {results[name]['error']}")
    
    # Save training summary
    summary_dir = Path(f'saved_models/KMRF/{classification_type}_labels/{asset_class}')
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary_path = summary_dir / f'training_summary_{end_date}.json'
    
    with open(summary_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ Training summary saved to: {summary_path}")
    print(f"{'='*80}\n")
    
    return results

## Run Batch Training

Execute this cell to train models for all configured asset classes:

In [24]:
# Run batch training for each selected asset class
all_results = {}

for asset_class in ASSET_CLASSES_TO_TRAIN:
    asset_names = ASSET_GROUPS.get(asset_class, [])
    
    if not asset_names:
        print(f"Warning: No assets defined for class '{asset_class}'. Skipping.")
        continue
    
    print(f"\n{'#'*80}")
    print(f"# TRAINING ASSET CLASS: {asset_class.upper()}")
    print(f"# Assets: {len(asset_names)}")
    print(f"{'#'*80}")
    
    results = train_kmrf_batch(
        asset_names=asset_names,
        asset_class=asset_class,
        classification_type=CLASSIFICATION_TYPE,
        end_date=END_DATE,
        validation_start=VALIDATION_START,
        validation_end=VALIDATION_END,
        test_start=TEST_START,
        use_boruta=USE_BORUTA,
        boruta_params={'max_iter': BORUTA_MAX_ITER},
        include_macro=INCLUDE_MACRO,
        random_seed=RANDOM_SEED,
        continue_on_error=CONTINUE_ON_ERROR,
        save_successful_only=SAVE_SUCCESSFUL_ONLY,
        verbose=VERBOSE
    )
    
    all_results[asset_class] = results

print(f"\n{'#'*80}")
print(f"# ALL BATCH TRAINING COMPLETE")
print(f"{'#'*80}")

# Overall summary
total_attempted = sum(len(results) for results in all_results.values())
total_successful = sum(
    sum(1 for r in results.values() if r['status'] == 'success')
    for results in all_results.values()
)
total_failed = total_attempted - total_successful

print(f"\nOverall Results:")
print(f"  Total Assets Attempted: {total_attempted}")
print(f"  Successful: {total_successful}")
print(f"  Failed: {total_failed}")
print(f"  Success Rate: {100*total_successful/total_attempted:.1f}%")


################################################################################
# TRAINING ASSET CLASS: COMMODITY
# Assets: 10
################################################################################

BATCH TRAINING KMRF MODELS
Asset Class: commodity
Classification Type: original
Total Assets: 10
Use Boruta: True
Include Macro: True
Training Period: up to 20190101
Validation: 2019-04-01 to 2019-09-30
Test: 2020-01-01 onwards


[1/10] Processing: Gold Futures

1. Initializing KMRF model...
KMRF model initialized
  Asset: Gold Futures
  Asset class: commodity
  Classification type: original
  End date: 20190101
  Using pre-computed features: True
  Data path: data/ready/commodity.csv
  KAMA+MSR model directory: saved_models/KAMA_MSR/commodity/20190101
  Validation period: 2019-04-01 to 2019-09-30
  Test start: 2020-01-01
  Random seed: 1010
2. Loading data...

Loading data from: data/ready/commodity.csv
Loaded data for: Gold Futures
  Rows: 10945
  Columns: 69
  Date range: 199